# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
def select_

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Endpoints product', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [19]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-5.1
Updated
1 day ago
•
35.9k
•
1.14k
openbmb/VoxCPM2
Updated
6 days ago
•
9.3k
•
810
google/gemma-4-31B-it
Updated
3 days ago
•
2.44M
•
1.83k
MiniMaxAI/MiniMax-M2.7
Updated
about 14 hours ago
•
18.3k
•
627
dealignai/Gemma-4-31B-JANG_4M-CRACK
Updated
4 days ago
•
107k
•
1.01k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
417
OmniVoice
🌍
417
High-quality voice cloning TTS for 600+ languages
Running
on
Zero
MCP
2.02k
Wan2.2 14B Preview
🐌
2.02k
generate a video from an image with a text prompt
Running
on
A10G
Featured
3

In [11]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-5.1\nUpdated\n1 day ago\n•\n35.9k\n•\n1.14k\nopenbmb/VoxCPM2\nUpdated\n6 days ago\n•\n9.3k\n•\n810\ngoogle/gemma-4-31B-it\nUpdated\n3 days ago\n•\n2.44M\n•\n1.83k\nMiniMaxAI/MiniMax-M2.7\nUpdated\nabout 14 hours ago\n•\n18.3k\n•\n627\ndealignai/Gemma-4-31B-JANG_4M-CRACK\nUpdated\n4 days ago\n•\n107k\n•\n1.01k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n418\nOmniV

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [23]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the premier collaboration platform for the global machine learning (ML) community, driving the future of artificial intelligence. It serves as a central hub where ML engineers, researchers, and enthusiasts can share, explore, and experiment with open-source models, datasets, and applications. Hugging Face empowers the next generation of AI talent to build an open, collaborative, and ethical AI ecosystem.

---

## Our Platform & Community

- **2M+ Models**: Access and contribute to a vast library of ML models spanning text, image, video, audio, and even 3D modalities.
- **500K+ Datasets**: Discover datasets from diverse fields, all available for exploration and use in ML projects.
- **Spaces**: Run, share, and interact with ML applications in a seamless environment.
- **Open Source & Collaboration**: Build your portfolio, collaborate openly, and gain recognition within a thriving community.
  
Hugging Face fosters a deeply connected AI community, continuously growing with contributions from leading organizations and individual researchers worldwide.

---

## Enterprise Solutions

Hugging Face offers scalable Team and Enterprise plans designed to meet organizational AI needs:

- **Security & Compliance**: Enterprise-grade security, Single Sign-On (SSO), detailed audit logs, and resource control.
- **Flexible Access Controls**: Granular repository permissions and centralized token management.
- **Advanced Analytics**: Single dashboard to track and analyze model and dataset usage.
- **Compute & Storage**: Enhanced compute options including ZeroGPU with quota boosts and private storage solutions.
- **Collaboration Tools**: Private dataset viewers and inference provider billing management to streamline workflows.

These features enable organizations to accelerate AI innovation securely and efficiently.

---

## Company Culture

- **Open and Ethical AI**: Hugging Face is committed to building an AI future that is transparent, ethical, and inclusive.
- **Community-Driven**: The company thrives on collaboration, knowledge sharing, and active engagement with its global user base.
- **Innovative & Inclusive**: Hugging Face's team includes talented scientists and engineers pushing the boundaries of AI technology.
- **Learning & Growth**: Encourages continuous learning and skill-building, supporting contributors to grow their portfolios and careers.

---

## Careers

Join Hugging Face to work at the heart of the AI revolution. The company values talent passionate about open-source technologies, collaboration, and ethical AI development. Team members benefit from a vibrant culture focused on innovation and community impact.

Explore current job openings and be part of a mission-driven organization shaping the future of machine learning.

---

## Connect & Explore

- **Website**: [huggingface.co](https://huggingface.co)
- **GitHub**: Explore open-source tools and libraries
- **Social Media**: Engage with the community on Twitter, LinkedIn, and Discord
- **Documentation & Support**: Access comprehensive docs and community forums to get started or advance your ML projects.

---

### Hugging Face — Building the Future of AI, Together.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


# Hugging Face Brochure

---

## Company Overview

**Hugging Face** is the vibrant AI community and collaboration platform dedicated to building the future of machine learning. It serves as a central hub where machine learning engineers, scientists, and enthusiasts worldwide share, explore, and experiment with open-source models, datasets, and applications. 

Hugging Face empowers the next generation of AI innovators to learn, collaborate, and contribute toward an open, ethical AI future.

---

## What We Offer

- **Extensive Model & Dataset Library:**  
  Access over 2 million models and 500,000 datasets across multiple modalities including text, image, video, audio, and even 3D.

- **Collaborative Platform:**  
  Host and collaborate on unlimited public models, datasets, and applications with the global machine learning community.

- **Open Source Stack:**  
  Rapidly move projects forward with a rich ecosystem of open-source libraries and tools designed for community use.

- **Spaces:**  
  Explore and run interactive AI applications and demos directly on the platform with support for community and enterprise deployments.

- **Compute and Enterprise Solutions:**  
  Accelerate machine learning workflows with paid compute resources and enterprise-grade tools for teams.

---

## Community and Culture

- **Inclusive and Ethical:**  
  Hugging Face fosters an open, ethical AI community committed to responsible machine learning development.

- **Collaborative Spirit:**  
  Encourages knowledge sharing through public repositories and collaborative spaces, helping individuals build portfolios and ML profiles recognized globally.

- **Global Reach:**  
  With a fast-growing and active community, Hugging Face supports contributors and users around the world, democratizing AI development.

---

## Who Uses Hugging Face?

Hugging Face is trusted by researchers, startups, large enterprises, and AI practitioners who leverage its tools and resources to:

- Develop state-of-the-art machine learning models.
- Share research and datasets openly.
- Deploy scalable AI applications.
- Innovate rapidly using a comprehensive open-source stack.

---

## Careers at Hugging Face

Join a forward-thinking organization at the forefront of AI technology. Hugging Face looks for passionate individuals who:

- Thrive in a collaborative and innovative environment.
- Are committed to open source and ethical AI.
- Wish to contribute to tools that empower thousands of AI practitioners globally.

Explore roles spanning engineering, research, community management, and enterprise services to build your career where technology meets community.

---

## Connect With Us

- **Website:** [huggingface.co](https://huggingface.co)  
- **Explore Models & Datasets:** Dive into ML resources and start building today.  
- **Join the Community:** Create an account and start collaborating with thousands of AI practitioners worldwide.

---

Build the future of AI — together with Hugging Face.  
*The home of machine learning collaboration.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>